Step 1 : Data Set Selection : HCV Data from UCI .

Step 2 : Machine Learning Classification Models and Evaluation metrics

        a. Data prepration and splitting

In [7]:
# ==============================================================================
# STEP 1: DATA PREPARATION & SPLITTING (LOCAL FILE VERSION)
# ==============================================================================
# Objective: Load raw data from directory, clean target into a binary task,
#            identify feature types, create a stratified train/test split,
#            and export 'test_data.csv' for the repository and Streamlit app.
# ==============================================================================

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. SPECIFY LOCAL FILE PATH
# ------------------------------------------------------------------------------
# Update the filename below if your file is named differently in the directory
file_path = "/hcvdat0.csv"

# Verify the file exists before reading
if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File '{file_path}' not found in the current directory! "
        )

df = pd.read_csv(file_path)

print("--- Raw Data Inspection ---")
print(f"Initial Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns present:", df.columns.tolist())

# 2. DROP REDUNDANT IDENTIFIER COLUMN
# ------------------------------------------------------------------------------
# 'Unnamed: 0' is a row index/ID column and carries no predictive value
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# 3. BINARIZE THE TARGET VARIABLE ('Category')
# ------------------------------------------------------------------------------
# Original categories in UCI dataset:
# - '0=Blood Donor', '0s=suspect Blood Donor' -> Healthy control group (0)
# - '1=Hepatitis', '2=Fibrosis', '3=Cirrhosis'  -> Liver disease patients (1)
disease_categories = ["1=Hepatitis", "2=Fibrosis", "3=Cirrhosis"]

# Map to binary target: 1 if patient has disease, 0 if healthy donor
df["target"] = df["Category"].apply(lambda val: 1 if str(val).strip() in disease_categories else 0)

# Drop original 'Category' text column to avoid target leakage
df = df.drop(columns=["Category"])

print("\n--- Target Variable Distribution ---")
print(f"Class 0 (Healthy): {(df['target'] == 0).sum()} samples")
print(f"Class 1 (Disease): {(df['target'] == 1).sum()} samples")

# 4. DEFINE INPUT FEATURES & TARGET
# ------------------------------------------------------------------------------
X = df.drop(columns=["target"])
y = df["target"]

# Categorical column needing encoding
categorical_cols = ["Sex"]

# Continuous numeric blood biomarkers + age needing imputation and scaling
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("\n--- Feature Breakdown ---")
print(f"Total Predictive Features: {len(X.columns)} (Minimum requirement: 12)")
print("Categorical Feature:", categorical_cols)
print("Numeric Biomarkers:", numeric_cols)

# 5. STRATIFIED TRAIN-TEST SPLIT (80% Train, 20% Test)
# ------------------------------------------------------------------------------
# Stratification ensures both train and test splits have identical class ratios.
# random_state=42 guarantees reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n--- Split Verification ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

# 6. EXPORT 'test_data.csv' FOR THE GITHUB REPO AND STREAMLIT APP
# ------------------------------------------------------------------------------
# Combines X_test features and the true target column into a standalone CSV
test_df = X_test.copy()
test_df["target"] = y_test
test_df.to_csv("test_data.csv", index=False)

print("\n[SUCCESS] 'test_data.csv' exported successfully to the local directory.")

--- Raw Data Inspection ---
Initial Shape: 615 rows, 14 columns
Columns present: ['Unnamed: 0', 'Category', 'Age', 'Sex', 'ALB', 'ALP', 'ALT', 'AST', 'BIL', 'CHE', 'CHOL', 'CREA', 'GGT', 'PROT']

--- Target Variable Distribution ---
Class 0 (Healthy): 540 samples
Class 1 (Disease): 75 samples

--- Feature Breakdown ---
Total Predictive Features: 12 (Minimum requirement: 12)
Categorical Feature: ['Sex']
Numeric Biomarkers: ['Age', 'ALB', 'ALP', 'ALT', 'AST', 'BIL', 'CHE', 'CHOL', 'CREA', 'GGT', 'PROT']

--- Split Verification ---
X_train shape: (492, 12)
X_test shape:  (123, 12)

[SUCCESS] 'test_data.csv' exported successfully to the local directory.


Step 2 : Machine Learning Classification Models and Evaluation metrics

       b. Preprocessing Setup & Pipeline Definition.

In [8]:
# ==============================================================================
# STEP 2: PREPROCESSING PIPELINES & DIRECTORY SETUP
# ==============================================================================
# Objective: Build a robust Scikit-Learn ColumnTransformer that handles:
#            1. Missing values (imputation)
#            2. Feature scaling for numeric biomarkers
#            3. Categorical encoding for 'Sex'
# ==============================================================================

import os
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. CREATE DIRECTORY FOR SAVED MODEL ARTIFACTS
# ------------------------------------------------------------------------------
# As required by Step 3 of the assignment: "model/ (saved trained models)"
os.makedirs("model", exist_ok=True)
print("[INFO] 'model/' directory verified/created.")

# 2. NUMERICAL FEATURE PIPELINE
# ------------------------------------------------------------------------------
# - SimpleImputer(strategy="median"):
#   Blood biomarkers (ALT, AST, Bilirubin, etc.) are heavily skewed by disease spikes.
#   The median is far more robust to extreme outlier spikes than the mean.
# - StandardScaler():
#   Standardizes features by removing the mean and scaling to unit variance (z-score).
#   This is mandatory for distance-sensitive algorithms (KNN, SVM, Logistic Regression).
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 3. CATEGORICAL FEATURE PIPELINE
# ------------------------------------------------------------------------------
# - SimpleImputer(strategy="most_frequent"):
#   Safely handles any potential missing categorical fields with the mode.
# - OneHotEncoder(drop="first", handle_unknown="ignore"):
#   Converts binary text ('m', 'f') into a clean numeric indicator (0 or 1).
#   'drop="first"' prevents multicollinearity for 2-category features.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# 4. BUNDLE INTO A SINGLE COLUMN TRANSFORMER
# ------------------------------------------------------------------------------
# This orchestrator directs each column to its designated sub-pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

print("[SUCCESS] Preprocessor defined successfully.")
print(f"Tracking {len(numeric_cols)} numeric columns and {len(categorical_cols)} categorical column.")

[INFO] 'model/' directory verified/created.
[SUCCESS] Preprocessor defined successfully.
Tracking 11 numeric columns and 1 categorical column.


Step 2 : Machine Learning Classification Models and Evaluation metrics

       c. Model Training and evaluation

In [9]:
# ==============================================================================
# STEP 3: MODEL TRAINING, EVALUATION, AND ARTIFACT SERIALIZATION
# ==============================================================================
# Objective: Train the 5 required classifiers, compute all 6 evaluation metrics
#            on the unseen test set, save trained pipelines as .joblib files,
#            and output the comparison table for the README.
# ==============================================================================

import joblib
import pandas as pd
import numpy as np

# Import the 5 required classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Import the 6 required evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# 1. INSTANTIATE THE 5 CLASSIFIERS
# ------------------------------------------------------------------------------
# Notes on configurations:
# - Logistic Regression: max_iter=1000 ensures gradient solver converges cleanly.
# - Decision Tree: max_depth=5 prevents deep, overfitted leaves on 492 rows.
# - KNN: n_neighbors=5 is an odd number to avoid tie-voting.
# - Gaussian NB: Selected because blood biomarker metrics are continuous values.
# - SVM: probability=True is required so the model can output predict_proba() for AUC.
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=42)
}

# 2. TRAIN, EVALUATE, AND EXPORT EACH MODEL
# ------------------------------------------------------------------------------
metrics_records = []

print("--- Training & Evaluation In Progress ---\n")

for name, clf in classifiers.items():
    # Glue preprocessing + classifier into an end-to-end pipeline
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", clf)
    ])

    # Fit ONLY on training data
    pipeline.fit(X_train, y_train)

    # Save the fitted pipeline artifact inside 'model/' directory
    # File names match the requirements of Step 3
    file_name = name.replace(" ", "_") + ".joblib"
    save_path = os.path.join("model", file_name)
    joblib.dump(pipeline, save_path)
    print(f"[SAVED] {name} artifact saved to: {save_path}")

    # Generate predictions on the unseen test set (X_test)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]  # Class 1 probabilities for AUC

    # Calculate all 6 required evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    metrics_records.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

# 3. DISPLAY THE COMPARISON TABLE
# ------------------------------------------------------------------------------
comparison_df = pd.DataFrame(metrics_records)
print("\n" + "="*70)
print(" EVALUATION METRICS COMPARISON TABLE")
print("="*70)
print(comparison_df.to_markdown(index=False))
print("="*70)

--- Training & Evaluation In Progress ---

[SAVED] Logistic Regression artifact saved to: model/Logistic_Regression.joblib
[SAVED] Decision Tree artifact saved to: model/Decision_Tree.joblib
[SAVED] KNN artifact saved to: model/KNN.joblib
[SAVED] Naive Bayes artifact saved to: model/Naive_Bayes.joblib
[SAVED] Support Vector Machine artifact saved to: model/Support_Vector_Machine.joblib

 EVALUATION METRICS COMPARISON TABLE
| ML Model Name          |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:-----------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression    |     0.9593 | 0.9228 |      1      |   0.6667 | 0.8    | 0.7982 |
| Decision Tree          |     0.9512 | 0.729  |      0.8462 |   0.7333 | 0.7857 | 0.7608 |
| KNN                    |     0.9431 | 0.9278 |      1      |   0.5333 | 0.6957 | 0.7077 |
| Naive Bayes            |     0.9106 | 0.9222 |      0.6429 |   0.6    | 0.6207 | 0.5705 |
| Support Vector Mach

Step 6 : Streamlit App

In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

st.set_page_config(page_title="HCV Classification Dashboard", layout="wide")

st.title("Hepatitis C Virus (HCV) - Classification & Model Evaluation")
st.markdown(
    "This application evaluates pre-trained classification models on an uploaded test dataset. "
    "Upload `test_data.csv` using the sidebar to view dynamic performance metrics and diagnostics."
)

# 1. Sidebar Controls
st.sidebar.header("Evaluation Controls")

uploaded_file = st.sidebar.file_uploader(
    "Upload Test Dataset (CSV)",
    type=["csv"],
    help="Upload test_data.csv containing all 12 input features and the true 'target' column."
)

model_paths = {
    "Logistic Regression": "model/Logistic_Regression.joblib",
    "Decision Tree": "model/Decision_Tree.joblib",
    "KNN": "model/KNN.joblib",
    "Naive Bayes": "model/Naive_Bayes.joblib",
    "Support Vector Machine": "model/Support_Vector_Machine.joblib"
}

selected_model_name = st.sidebar.selectbox(
    "Select Classification Model",
    list(model_paths.keys())
)

TARGET_COLUMN = "target"

# 2. Evaluation Logic
if uploaded_file is not None:
    try:
        test_df = pd.read_csv(uploaded_file)

        if TARGET_COLUMN not in test_df.columns:
            st.error(f"Error: Target column '{TARGET_COLUMN}' not found in the uploaded file.")
        else:
            X_eval = test_df.drop(columns=[TARGET_COLUMN])
            y_eval = test_df[TARGET_COLUMN]

            chosen_artifact_path = model_paths[selected_model_name]

            if not os.path.exists(chosen_artifact_path):
                st.error(f"Model file not found at: {chosen_artifact_path}. Please check your 'model/' directory.")
            else:
                pipeline = joblib.load(chosen_artifact_path)

                y_pred = pipeline.predict(X_eval)
                y_proba = pipeline.predict_proba(X_eval)[:, 1] if hasattr(pipeline, "predict_proba") else None

                acc = accuracy_score(y_eval, y_pred)
                auc = roc_auc_score(y_eval, y_proba) if y_proba is not None else np.nan
                prec = precision_score(y_eval, y_pred, zero_division=0)
                rec = recall_score(y_eval, y_pred, zero_division=0)
                f1 = f1_score(y_eval, y_pred, zero_division=0)
                mcc = matthews_corrcoef(y_eval, y_pred)

                st.subheader(f"Evaluation Metrics: {selected_model_name}")
                col1, col2, col3, col4, col5, col6 = st.columns(6)
                col1.metric("Accuracy", f"{acc:.4f}")
                col2.metric("AUC Score", f"{auc:.4f}" if not np.isnan(auc) else "N/A")
                col3.metric("Precision", f"{prec:.4f}")
                col4.metric("Recall", f"{rec:.4f}")
                col5.metric("F1 Score", f"{f1:.4f}")
                col6.metric("MCC Score", f"{mcc:.4f}")

                st.divider()

                col_left, col_right = st.columns(2)

                with col_left:
                    st.subheader("Confusion Matrix")
                    cm = confusion_matrix(y_eval, y_pred)
                    fig, ax = plt.subplots(figsize=(4.5, 3.5))
                    sns.heatmap(
                        cm,
                        annot=True,
                        fmt="d",
                        cmap="Blues",
                        xticklabels=["Healthy (0)", "Disease (1)"],
                        yticklabels=["Healthy (0)", "Disease (1)"],
                        ax=ax
                    )
                    ax.set_xlabel("Predicted Label")
                    ax.set_ylabel("True Label")
                    st.pyplot(fig)
                    plt.close(fig)

                with col_right:
                    st.subheader("Classification Report")
                    report_dict = classification_report(
                        y_eval,
                        y_pred,
                        target_names=["Healthy (0)", "Disease (1)"],
                        output_dict=True
                    )
                    report_df = pd.DataFrame(report_dict).transpose()
                    st.dataframe(report_df.style.format("{:.3f}"), use_container_width=True)

    except Exception as e:
        st.error(f"An error occurred while evaluating the dataset: {e}")

else:
    st.info("Please upload `test_data.csv` using the sidebar on the left to view model evaluation results.")

Overwriting app.py


Step 4 : Requirements Txt

In [11]:
%%writefile requirements.txt
streamlit>=1.30.0
scikit-learn>=1.3.0
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0

Overwriting requirements.txt


Step 5 : Readme File

In [12]:
# Run this in Colab to create the exact README file
readme_content = """# Hepatitis C Virus (HCV) Patient Classification

## a. Problem statement
The objective of this project is to develop and evaluate five supervised machine learning classification models to predict whether an individual has liver disease (Hepatitis, Fibrosis, or Cirrhosis) or is a healthy control based on standard laboratory blood biomarkers and demographic factors.

Early, non-invasive detection of hepatic pathology using routine biochemical measurements is critical in clinical settings to identify patients at risk before irreversible liver fibrosis, end-stage cirrhosis, or liver failure occurs.

---

## b. Dataset description
* **Source:** UCI Machine Learning Repository (HCV Data, Dataset ID: 571)
* **Instance Count:** 615 samples
* **Predictive Features:** 12 total features
  * **Demographics (2 features):**
    * `Age`: Age of the subject in years (continuous numeric)
    * `Sex`: Biological sex (categorical: 'm', 'f')
  * **Laboratory Blood Biomarkers (10 continuous numeric features):**
    * `ALB`: Albumin (g/L)
    * `ALP`: Alkaline Phosphatase (IU/L)
    * `ALT`: Alanine Transaminase (U/L)
    * `AST`: Aspartate Transaminase (U/L)
    * `BIL`: Bilirubin (μmol/L)
    * `CHE`: Cholinesterase (kU/L)
    * `CHOL`: Cholesterol (mmol/L)
    * `CREA`: Creatinine (μmol/L)
    * `GGT`: Gamma-Glutamyl Transferase (U/L)
    * `PROT`: Total Protein (g/L)
* **Target Formulation (`target`):**
  * `0` (Healthy Control): Derived from original categories `0=Blood Donor` and `0s=suspect Blood Donor` (540 total samples)
  * `1` (Liver Disease): Derived from categories `1=Hepatitis`, `2=Fibrosis`, and `3=Cirrhosis` (75 total samples)
* **Preprocessing Pipeline:**
  * Dropped the uninformative patient identifier column (`Unnamed: 0`).
  * Imputed missing values in continuous blood biomarkers using the feature median to resist skew from clinical outlier spikes.
  * Encoded categorical `Sex` via One-Hot Encoding (`drop='first'`).
  * Scaled continuous numeric biomarkers using `StandardScaler` (z-score normalization).
  * Applied an 80/20 stratified train-test split (492 train instances, 123 test instances) to maintain the exact ~88% to ~12% class balance on unseen evaluation data.

---

## c. Github Repository Link
* **GitHub Repository URL:** [INSERT_YOUR_GITHUB_REPO_URL_HERE]
* **Streamlit Live Deployment URL:** [INSERT_YOUR_STREAMLIT_APP_URL_HERE]

---

## d. Models used:

All five classifiers were fitted on the training split (492 instances) and evaluated on the exact same unseen test split (`test_data.csv`, 123 instances).

### Comparison Table

| ML Model Name | Accuracy | AUC | Precision | Recall | F1 | MCC |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| Logistic Regression | 0.9593 | 0.9228 | 1.0000 | 0.6667 | 0.8000 | 0.7982 |
| Decision Tree | 0.9512 | 0.7290 | 0.8462 | 0.7333 | 0.7857 | 0.7608 |
| KNN | 0.9431 | 0.9278 | 1.0000 | 0.5333 | 0.6957 | 0.7077 |
| Naive Bayes | 0.9106 | 0.9222 | 0.6429 | 0.6000 | 0.6207 | 0.5705 |
| Support Vector Machine | 0.9350 | 0.9802 | 0.7692 | 0.6667 | 0.7143 | 0.6800 |

---

### Observations on Model Performance

| ML Model Name | Observation about model performance |
| :--- | :--- |
| **Logistic Regression** | Achieved the highest overall Accuracy (0.9593), F1 score (0.8000), and MCC (0.7982) with zero False Positives (Precision = 1.0000). The standardized biomarker boundaries allow the linear hyperplane to cleanly isolate unambiguous healthy controls, though it missed one-third of disease cases (Recall = 0.6667). |
| **Decision Tree** | Produced the highest Recall among all tested models (0.7333), capturing non-linear threshold triggers (e.g., elevated AST paired with high BIL). However, its AUC (0.7290) was the lowest because unpruned probability steps yield crude, step-like discrimination thresholds rather than smooth probabilistic curves. |
| **KNN** | Demonstrated perfect Precision (1.0000) and a competitive AUC (0.9278), but achieved the lowest Recall (0.5333). In a 12-dimensional feature space with severe class imbalance (88% healthy), majority-class nearest neighbors dominate localized voting spheres around peripheral disease instances. |
| **Naive Bayes** | Showed the weakest overall performance across Accuracy (0.9106), F1 (0.6207), and MCC (0.5705). The algorithm's fundamental assumption of conditional feature independence is heavily violated by clinical biomarkers; liver enzymes like ALT, AST, and GGT exhibit strong mutual collinearity during hepatic cell damage. |
| **Support Vector Machine** | Achieved the highest discrimination capacity with an outstanding AUC of 0.9802 using the RBF kernel. It successfully maps non-linear enzyme interactions into higher dimensions, maintaining stable trade-offs across precision (0.7692) and recall (0.6667). |
| **Overall, Winner for your dataset?** | **Logistic Regression** is the overall top performer based on the highest balanced metric scores (F1: 0.8000, MCC: 0.7982, Accuracy: 0.9593). However, for practical clinical diagnosis where detecting sick patients is paramount, **Support Vector Machine (SVM)** is a strong co-winner due to its superior AUC (0.9802), allowing clinicians to adjust the probability threshold to prioritize Recall. |
"""

with open("README.md", "w") as f:
    f.write(readme_content)

from google.colab import files
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>